# 5. Downstream Analysis

On top of predicted ST expression, `stp.downstream(mode=...)` runs three
biologically-oriented analyses and, by default, compares each against the
same analysis run on ground-truth ST — so you see not just per-gene
accuracy, but whether biologically meaningful structure survives
prediction.

| Mode | What it does | Compared against ground truth via |
|---|---|---|
| `"gene_enrichment"` | Pathway activity scoring (ssGSEA / rank-based) via [gseapy](https://github.com/zqfang/GSEApy) | pathway-by-pathway correlation |
| `"deconvolution"` | Per-spot cell-type abundance via [cell2location](https://github.com/BayraktarLab/cell2location) | cell-type-by-cell-type correlation |
| `"spatial_domain"` | Spatial domain clustering via [SpaGCN](https://github.com/jianhuupenn/SpaGCN) | ARI / NMI / AMI + Hungarian-matched label accuracy |


> **Prerequisites**
> - STP-Bench installed (`bash scripts/create_env.sh`, see the
>   [README](../README.md#installation)).
> - Benchmark data downloaded for the dataset(s) used below (see
>   [README — Benchmark Data](../README.md#benchmark-data)), or your own
>   dataset added following
>   [docs/guide.md — Adding a New Dataset](../docs/guide.md#adding-a-new-dataset).
> - Run this notebook from the repo root, or pass `repo_root=` explicitly to
>   `STPred(...)`.
>
> `deconvolution` / `spatial_domain` additionally need heavier dependencies
> not installed by default:
> ```bash
> uv pip install -r requirements/downstream.txt
> ```


In [ ]:
from stpbench import STPred

stp = STPred(models=["LinearProb"], repo_root=".")
stp.preprocess(data="ncche/xenium")
stp.train(data="ncche/xenium")
eval_res = stp.evaluate_internal(data="ncche/xenium")


## Gene-set enrichment

Pass the prior evaluation/prediction result directly — `downstream()` reads
the same prediction paths it already knows about, no need to re-specify
`data`/`train_data`/`folds`.


In [ ]:
enrichment = stp.downstream(mode="gene_enrichment", prior_result=eval_res)
enrichment.summary()
enrichment.save("gene_enrichment_metrics.csv")


By default, `gene_enrichment`'s pathway library (e.g.
`MSigDB_Hallmark_2020`) is fetched automatically from Enrichr via `gseapy`
and cached under `DATA.downstream.gene_enrichment.cache_dir` — point
`library` at a local `.gmt` file instead on a machine with no internet
access, via `overrides={"library": "/path/to/library.gmt"}`.


## Spatial domain clustering

Needs no external reference — clusters directly on predicted/ground-truth
expression and spatial coordinates.


In [ ]:
domains = stp.downstream(mode="spatial_domain", prior_result=eval_res)
domains.summary()


## Cell-type deconvolution

Needs a labeled single-cell reference atlas for the tissue of interest
(`.obs[labels_key]` cell-type labels, `.obs[batch_key]` batch,
`.var['feature_name']` gene symbols, `.layers['count']` raw counts) — pass
its path via `overrides`, or set it once per dataset under
`DATA.downstream.deconvolution.reference_path` in the data config.


In [ ]:
deconv = stp.downstream(
    mode="deconvolution",
    prior_result=eval_res,
    overrides={"reference_path": "/path/to/single_cell_reference.h5ad"},
)
deconv.summary()


## Running without a prior result

`prior_result` is a convenience — you can instead give `data`/`train_data`/
`folds` directly and `downstream()` recomputes the same prediction-path
convention `evaluate()` itself uses:


In [ ]:
stp.downstream(mode="gene_enrichment", data="ncche/xenium", train_data="ncche/xenium")


## Predictions only, no ground truth

Set `evaluate_against_gt=False` to run the analysis purely on predicted
expression — useful for the WSI-only inference targets from notebook 3,
which have no ground truth to compare against.


In [ ]:
stp.downstream(mode="gene_enrichment", prior_result=eval_res, evaluate_against_gt=False)


## Output layout

Extends the existing prediction path:

```
<DATA.output_dir>/<data>/<model>/[<train_data>/]fold<k>/downstream/<mode>/
    <sample>.csv / <sample>.h5ad     # analysis run on PREDICTED expression
    eval/metrics.csv                 # comparison vs. ground truth
    eval/gt/                         # cached ground-truth-side results
```

Mode-specific hyperparameter defaults live in
`config/downstream/defaults.yaml`; override per dataset under
`DATA.downstream.<mode>` in the data config, or per call via
`downstream(..., overrides={...})` as shown above.
